# Band-Pass Wave Stacking — 140°W (TPOSE24 Ri5)

Compares TAO ADCP vs TPOSE24 (Ri5) zonal velocity at 0°N/140°W, and shows model-only at 1°N and 1°S for three frequency bands:
- **Band 1:** >6.5-day periods (low-frequency)
- **Band 2:** 1.2–6.5 day periods
- **Band 3:** 21–29-hour periods (near-diurnal / near-inertial)

**Figure 1** (main): 3 rows × 2 cols — left=TAO, right=TPOSE at 0°N  
**Figure 2**: 3 rows × 2 cols — left=TPOSE 1°N, right=TPOSE 1°S

In [1]:
SUBFOLDER = '3month_Ri5'
RUN_DIR   = f'/data/SO3/edavenport/tpose24/oct2012_TP6Vel_{SUBFOLDER}'
PROF_FILE = f'{RUN_DIR}/PROF_eq/TAO_WO_2012_ADCP_v2_model.nc'
SAVE_DIR  = SUBFOLDER   # relative to wave_stacking/ (notebook cwd)
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cmocean.cm as cmo
import pandas as pd
import warnings
from scipy.signal import butter, filtfilt

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10})

## 1. Load TAO + MITprof model at 0°N, 140°W (hourly)

In [ ]:
ds         = xr.open_dataset(PROF_FILE)
lons       = ds['prof_lon'].values
lats       = ds['prof_lat'].values
depths_tao = ds['prof_depth'].values        # (62,)  10–315 m
dates_raw  = ds['prof_YYYYMMDD'].values
times_raw  = ds['prof_HHMMSS'].values
U_all      = ds['prof_U'].values.astype(float)
Ue_all     = ds['prof_Uestim'].values.astype(float)
V_all      = ds['prof_V'].values.astype(float)
Ve_all     = ds['prof_Vestim'].values.astype(float)

def parse_dt(d, t):
    d, t = int(d), int(t)
    return pd.Timestamp(d // 10000, (d % 10000) // 100, d % 100,
                        t // 10000, (t % 10000) // 100, t % 100)

timestamps = np.array([parse_dt(d, t) for d, t in zip(dates_raw, times_raw)])

# 0°N, 140°W only; require model output
mask  = (lons == -140.) & (lats == 0.)
ok    = ~np.isnan(Ue_all[mask]).any(axis=1)
t_raw = timestamps[mask][ok]
U_raw = U_all[mask][ok]
Ue_raw= Ue_all[mask][ok]
V_raw = V_all[mask][ok]
Ve_raw= Ve_all[mask][ok]

# Regular hourly grid
t_hrly = pd.date_range(t_raw[0].floor('h'), t_raw[-1].ceil('h'), freq='1h')
n_hrly, n_z = len(t_hrly), len(depths_tao)
U_tao  = np.full((n_hrly, n_z), np.nan)
Ue_0N  = np.full((n_hrly, n_z), np.nan)
V_tao  = np.full((n_hrly, n_z), np.nan)
Ve_0N  = np.full((n_hrly, n_z), np.nan)

for i, tr in enumerate(t_raw):
    idx = int(round((tr - t_hrly[0]).total_seconds() / 3600))
    if 0 <= idx < n_hrly:
        U_tao[idx]  = U_raw[i]
        Ue_0N[idx]  = Ue_raw[i]
        V_tao[idx]  = V_raw[i]
        Ve_0N[idx]  = Ve_raw[i]

Z_tao = -depths_tao
print(f'TAO: {n_hrly} hourly steps,  {t_hrly[0].date()} – {t_hrly[-1].date()}')
print(f'Depths: {depths_tao[0]:.0f}–{depths_tao[-1]:.0f} m  ({n_z} levels)')

## 2. Load TPOSE model at 0°N, 1°N, 1°S (3-hourly)

Reads from a pre-built cache NetCDF.  
If the cache is missing, run `build_model_cache.py` first (~15 min on NFS).

In [ ]:
cache_u = f'{SAVE_DIR}/cache_model_points.nc'
cache_v = f'{SAVE_DIR}/cache_model_points_V.nc'

for cf in [cache_u, cache_v]:
    if not os.path.exists(cf):
        raise FileNotFoundError(
            f'Cache not found: {cf}\n'
            f'Run wave_stacking/build_model_cache.py {SUBFOLDER} (U) or '
            f'build_v_cache.py {SUBFOLDER} (V) first.')

print('Loading U cache …')
ds_u     = xr.open_dataset(cache_u)
U_0N_mod = ds_u['U_0N'].values
U_1N_mod = ds_u['U_1N'].values
U_1S_mod = ds_u['U_1S'].values
t_mod    = pd.DatetimeIndex(ds_u['time'].values)
Z_mod    = ds_u['depth'].values

print('Loading V cache …')
ds_v     = xr.open_dataset(cache_v)
V_0N_mod = ds_v['V_0N'].values
V_1N_mod = ds_v['V_1N'].values
V_1S_mod = ds_v['V_1S'].values

print(f'Model: {len(t_mod)} 3-hourly steps,  {t_mod[0].date()} – {t_mod[-1].date()}')
print(f'Depths: {-Z_mod[0]:.1f}–{-Z_mod[-1]:.1f} m  ({len(Z_mod)} levels)')

## 3. Band-pass filter function

In [5]:
def bpfilt(data_2d, dt_h, band_periods_h, order=4):
    """
    Apply zero-phase Butterworth filter to each depth column independently.

    band_periods_h : tuple
        (T_hi,)       → low-pass  (keeps periods > T_hi)
        (T_lo, T_hi)  → band-pass (keeps T_lo < period < T_hi)
        Periods in hours.
    NaN columns skipped; gaps linearly interpolated before filtering, then masked back.
    """
    fs  = 1.0 / dt_h
    nyq = fs / 2.0

    if len(band_periods_h) == 1:
        Wn = min((1.0 / band_periods_h[0]) / nyq, 0.99)
        b, a = butter(order, Wn, btype='low')
    else:
        T_lo, T_hi = band_periods_h
        Wn = [max((1.0 / T_hi) / nyq, 1e-4),
              min((1.0 / T_lo) / nyq, 0.99)]
        b, a = butter(order, Wn, btype='band')

    out = np.full_like(data_2d, np.nan)
    t   = np.arange(data_2d.shape[0])
    min_pts = max(60, 6 * int(max(band_periods_h) / dt_h))

    for iz in range(data_2d.shape[1]):
        col = data_2d[:, iz]
        ok  = np.isfinite(col)
        if ok.sum() < min_pts:
            continue
        col_f = np.interp(t, t[ok], col[ok])
        col_f -= col_f.mean()
        try:
            filt = filtfilt(b, a, col_f)
        except Exception:
            continue
        filt[~ok] = np.nan
        out[:, iz] = filt
    return out

## 4. Apply filters

In [ ]:
BANDS = [
    ('>6.5-day',    (6.5 * 24,)),           # lowpass at 156 h
    ('1.2–6.5-day', (1.2 * 24, 6.5 * 24)), # bandpass 28.8–156 h
    ('21–29-hour',  (21., 29.)),             # bandpass 21–29 h
    ('9–15-hour',   (9., 15.)),              # semidiurnal
]

print('Filtering U …')
filt_tao_u = [bpfilt(U_tao,     1.0, bp) for _, bp in BANDS]
filt_0N_u  = [bpfilt(Ue_0N,    1.0, bp) for _, bp in BANDS]
filt_1N_u  = [bpfilt(U_1N_mod, 3.0, bp) for _, bp in BANDS]
filt_1S_u  = [bpfilt(U_1S_mod, 3.0, bp) for _, bp in BANDS]

print('Filtering V …')
filt_tao_v = [bpfilt(V_tao,     1.0, bp) for _, bp in BANDS]
filt_0N_v  = [bpfilt(Ve_0N,    1.0, bp) for _, bp in BANDS]
filt_1N_v  = [bpfilt(V_1N_mod, 3.0, bp) for _, bp in BANDS]
filt_1S_v  = [bpfilt(V_1S_mod, 3.0, bp) for _, bp in BANDS]

# Mask TPOSE 0N wherever TAO obs is NaN (point-by-point)
tao_u_nan = np.isnan(U_tao)
tao_v_nan = np.isnan(V_tao)
for i in range(len(BANDS)):
    filt_0N_u[i] = np.where(tao_u_nan, np.nan, filt_0N_u[i])
    filt_0N_v[i] = np.where(tao_v_nan, np.nan, filt_0N_v[i])

print('Done.')

## 5. Plotting helpers

In [ ]:
import matplotlib.ticker as mticker

def fmt_time_ax(ax, day_interval=None):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    if day_interval:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=day_interval))
    else:
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

def depth_yax(ax):
    ax.set_ylim([-315, 0])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{-y:.0f}'))
    ax.set_ylabel('Depth (m)')

def cf_plot(ax, t_arr, z_arr, data, vmax, n_levels=100):
    levels = np.linspace(-vmax, vmax, n_levels + 1)
    return ax.contourf(t_arr, z_arr, data.T, levels=levels,
                       cmap=cmo.balance, extend='both')

def make_colorbar(fig, cf, ax_list, vmax):
    """Shared colorbar with ~5–7 sensible ticks."""
    cb = fig.colorbar(cf, ax=ax_list, shrink=0.85, pad=0.01)
    cb.set_label('m s⁻¹')
    step = _nice_tick(vmax / 3)
    ticks = np.arange(-vmax, vmax + step / 2, step)
    cb.set_ticks(ticks[np.abs(ticks) <= vmax * 1.01])
    cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3g'))

def _nice_tick(x):
    import math
    exp  = math.floor(math.log10(x)) if x > 0 else 0
    frac = x / 10 ** exp
    for n in [1, 2, 2.5, 5, 10]:
        if frac <= n:
            return n * 10 ** exp
    return 10 ** (exp + 1)

def vmax_of(*arrays):
    vals = np.concatenate([a[np.isfinite(a)] for a in arrays])
    return max(np.nanpercentile(np.abs(vals), 97), 0.005)

## 6. Figure 1 — TAO vs TPOSE at 0°N, 140°W  (full period)

In [ ]:
t_hrly_arr = t_hrly.to_pydatetime()

def plot_0N_fig(filt_tu, filt_mu, filt_tv, filt_mv, vmaxes=None,
                xlim=None, day_interval=None, title=''):
    """4 rows × 4 cols: TAO U | TPOSE U | TAO V | TPOSE V.  One shared colorbar per row."""
    fig, axes = plt.subplots(4, 4, figsize=(22, 14),
                              sharex=True, sharey=True, constrained_layout=True)
    fig.suptitle(title, fontsize=11)
    vmaxes_out = []
    for row, (label, _) in enumerate(BANDS):
        vm = vmaxes[row] if vmaxes else vmax_of(filt_tu[row], filt_mu[row],
                                                 filt_tv[row], filt_mv[row])
        vmaxes_out.append(vm)

        cf_plot(axes[row, 0], t_hrly_arr, Z_tao, filt_tu[row], vm)
        cf_plot(axes[row, 1], t_hrly_arr, Z_tao, filt_mu[row], vm)
        cf_plot(axes[row, 2], t_hrly_arr, Z_tao, filt_tv[row], vm)
        cf = cf_plot(axes[row, 3], t_hrly_arr, Z_tao, filt_mv[row], vm)

        axes[row, 0].set_title(f'TAO U  [{label}]')
        axes[row, 1].set_title(f'TPOSE U  [{label}]')
        axes[row, 2].set_title(f'TAO V  [{label}]')
        axes[row, 3].set_title(f'TPOSE V  [{label}]')
        for ax in axes[row]:
            depth_yax(ax)
            if xlim:
                ax.set_xlim(xlim)

        make_colorbar(fig, cf, list(axes[row, :]), vm)

    for ax in axes[-1]:
        fmt_time_ax(ax, day_interval=day_interval)
    return fig, vmaxes_out

fig1, vmaxes_fig1 = plot_0N_fig(
    filt_tao_u, filt_0N_u, filt_tao_v, filt_0N_v,
    title=f'Band-Filtered Velocity — 0°N, 140°W  ({SUBFOLDER})  Oct–Dec 2012')

out1 = f'{SAVE_DIR}/fig1_wave_stacking_0N_140W.png'
fig1.savefig(out1, dpi=150, bbox_inches='tight')
plt.show()
print('Saved', out1)

## 7. Figure 2 — TPOSE 1°N vs 1°S, 140°W  (full period)

In [ ]:
t_mod_arr = t_mod.to_pydatetime()

def plot_1N1S_fig(filt_1nu, filt_1nv, filt_1su, filt_1sv, vmaxes=None,
                  vmax_overrides=None, xlim=None, day_interval=None, title=''):
    """4 rows × 4 cols: 1N U | 1N V | 1S U | 1S V.  One shared colorbar per row.
    vmax_overrides: dict {row_index: vmax} to force specific rows to a given scale.
    """
    fig, axes = plt.subplots(4, 4, figsize=(22, 14),
                              sharex=True, sharey=True, constrained_layout=True)
    fig.suptitle(title, fontsize=11)
    vmaxes_out = []
    for row, (label, _) in enumerate(BANDS):
        if vmaxes:
            vm = vmaxes[row]
        else:
            vm = vmax_of(filt_1nu[row], filt_1nv[row], filt_1su[row], filt_1sv[row])
        if vmax_overrides and row in vmax_overrides:
            vm = vmax_overrides[row]
        vmaxes_out.append(vm)

        cf_plot(axes[row, 0], t_mod_arr, Z_mod, filt_1nu[row], vm)
        cf_plot(axes[row, 1], t_mod_arr, Z_mod, filt_1nv[row], vm)
        cf_plot(axes[row, 2], t_mod_arr, Z_mod, filt_1su[row], vm)
        cf = cf_plot(axes[row, 3], t_mod_arr, Z_mod, filt_1sv[row], vm)

        axes[row, 0].set_title(f'1°N U  [{label}]')
        axes[row, 1].set_title(f'1°N V  [{label}]')
        axes[row, 2].set_title(f'1°S U  [{label}]')
        axes[row, 3].set_title(f'1°S V  [{label}]')
        for ax in axes[row]:
            depth_yax(ax)
            if xlim:
                ax.set_xlim(xlim)

        make_colorbar(fig, cf, list(axes[row, :]), vm)

    for ax in axes[-1]:
        fmt_time_ax(ax, day_interval=day_interval)
    return fig, vmaxes_out

# Semidiurnal row (index 3) shares the colorbar from the 0°N TAO/TPOSE figure
fig2, vmaxes_fig2 = plot_1N1S_fig(
    filt_1N_u, filt_1N_v, filt_1S_u, filt_1S_v,
    vmax_overrides={3: vmaxes_fig1[3]},
    title=f'Band-Filtered Velocity — TPOSE24 {SUBFOLDER} | 140°W  Oct–Dec 2012')

out2 = f'{SAVE_DIR}/fig2_wave_stacking_1N_1S_140W.png'
fig2.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print('Saved', out2)

## 8. Figures 3 & 4 — Zoom: Nov 25 – Dec 15

In [ ]:
ZOOM_START = pd.Timestamp('2012-11-25')
ZOOM_END   = pd.Timestamp('2012-12-15')
xlim1      = [ZOOM_START, ZOOM_END]

fig3, _ = plot_0N_fig(
    filt_tao_u, filt_0N_u, filt_tao_v, filt_0N_v,
    vmaxes=vmaxes_fig1, xlim=xlim1, day_interval=3,
    title=f'Band-Filtered Velocity — 0°N, 140°W  ({SUBFOLDER})  Nov 25–Dec 15 2012')
out3 = f'{SAVE_DIR}/fig3_wave_stacking_0N_140W_zoom.png'
fig3.savefig(out3, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out3)

fig4, _ = plot_1N1S_fig(
    filt_1N_u, filt_1N_v, filt_1S_u, filt_1S_v,
    vmaxes=vmaxes_fig2, xlim=xlim1, day_interval=3,
    title=f'Band-Filtered Velocity — TPOSE24 {SUBFOLDER} | 140°W  Nov 25–Dec 15 2012')
out4 = f'{SAVE_DIR}/fig4_wave_stacking_1N_1S_140W_zoom.png'
fig4.savefig(out4, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out4)

## 9. Figures 5 & 6 — Zoom: Nov 25 – Dec 2

In [ ]:
ZOOM2_START = pd.Timestamp('2012-11-25')
ZOOM2_END   = pd.Timestamp('2012-12-02')
xlim2       = [ZOOM2_START, ZOOM2_END]

fig5, _ = plot_0N_fig(
    filt_tao_u, filt_0N_u, filt_tao_v, filt_0N_v,
    vmaxes=vmaxes_fig1, xlim=xlim2, day_interval=2,
    title=f'Band-Filtered Velocity — 0°N, 140°W  ({SUBFOLDER})  Nov 25–Dec 2 2012')
out5 = f'{SAVE_DIR}/fig5_wave_stacking_0N_140W_zoom2.png'
fig5.savefig(out5, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out5)

fig6, _ = plot_1N1S_fig(
    filt_1N_u, filt_1N_v, filt_1S_u, filt_1S_v,
    vmaxes=vmaxes_fig2, xlim=xlim2, day_interval=2,
    title=f'Band-Filtered Velocity — TPOSE24 {SUBFOLDER} | 140°W  Nov 25–Dec 2 2012')
out6 = f'{SAVE_DIR}/fig6_wave_stacking_1N_1S_140W_zoom2.png'
fig6.savefig(out6, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out6)

## 10. Figures 7 & 8 — Zoom: first 8 days (Oct 1–8)

In [ ]:
ZOOM3_START = t_hrly[0].to_pydatetime()
ZOOM3_END   = (t_hrly[0] + pd.Timedelta(days=8)).to_pydatetime()
xlim3       = [ZOOM3_START, ZOOM3_END]

fig7, _ = plot_0N_fig(
    filt_tao_u, filt_0N_u, filt_tao_v, filt_0N_v,
    vmaxes=vmaxes_fig1, xlim=xlim3, day_interval=2,
    title=f'Band-Filtered Velocity — 0°N, 140°W  ({SUBFOLDER})  First 8 Days')
out7 = f'{SAVE_DIR}/fig7_wave_stacking_0N_140W_zoom3.png'
fig7.savefig(out7, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out7)

fig8, _ = plot_1N1S_fig(
    filt_1N_u, filt_1N_v, filt_1S_u, filt_1S_v,
    vmaxes=vmaxes_fig2, xlim=xlim3, day_interval=2,
    title=f'Band-Filtered Velocity — TPOSE24 {SUBFOLDER} | 140°W  First 8 Days')
out8 = f'{SAVE_DIR}/fig8_wave_stacking_1N_1S_140W_zoom3.png'
fig8.savefig(out8, dpi=150, bbox_inches='tight')
plt.show(); print('Saved', out8)